# Person 1: train and hand off the dense baseline

Run these cells in order in Google Colab. Select **Runtime → Change runtime type → T4 GPU** (or another available CUDA GPU), then connect. Record the actual GPU and use the same device/precision for Karan's timing comparison.

You need **person1_colab_bundle.zip**. It contains the fixed real TinyStories data, tokenizer and code. The local CPU pilot is preliminary; this notebook runs the planned CUDA baseline. The notebook has been structurally checked locally; CUDA execution still needs to run in your Colab session.

The main run uses 5000 updates. A separate 2500-update fallback is available after inspecting pilot throughput. Both preserve effective batch 32 and context 128. Final-test data is never evaluated here.

**Checkpoint persistence:** checkpoints and logs go to your Google Drive folder. Mounting Drive requires your authorization. When restarting a disconnected runtime, rerun setup and use the resume cell below.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/MoE_Person1")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent outputs:", DRIVE_ROOT)

## Load the project bundle

On the first run, select the supplied ZIP when prompted. Later sessions can reuse the copy in Drive. Set UPLOAD_NEW_BUNDLE to True if you have a newer code/data bundle.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from google.colab import files
UPLOAD_NEW_BUNDLE = False
archive_path = DRIVE_ROOT / "person1_colab_bundle.zip"
if UPLOAD_NEW_BUNDLE or not archive_path.exists():
    uploaded = files.upload()
    matches = [name for name in uploaded if name.endswith(".zip")]
    if len(matches) != 1:
        raise ValueError("Upload exactly one project ZIP")
    archive_path.write_bytes(uploaded[matches[0]])
PROJECT = Path("/content/DeepLearningProject")
fresh_extraction = not PROJECT.exists()
if fresh_extraction:
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (Path("/content") / member.filename).resolve()
            if not target.is_relative_to(PROJECT):
                raise ValueError("Unexpected archive path")
        archive.extractall("/content")
manifest = json.loads((PROJECT / "bundle_manifest.json").read_text())
for name, expected in manifest.items():
    if not fresh_extraction and not name.startswith(("src/", "configs/", "data/")):
        continue
    assert hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() == expected, name
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(PROJECT / "src")
os.environ["MPLCONFIGDIR"] = str(PROJECT / "tmp/matplotlib")
sys.path.insert(0, str(PROJECT / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

def run_module(name, *args):
    subprocess.run([sys.executable, "-u", "-m", name, *map(str, args)], check=True)

print("Bundle checksums verified; project ready:", PROJECT)

## Verify GPU and pipeline

These checks cover data integrity, causality, input/target shifting, checkpoint reload and reproducible resumed training. The repeated-batch check is only a correctness check.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before training."
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
run_module("moe_transformer.checks", "--config", "configs/milestone.yaml",
           "--device", "cuda", "--output", DRIVE_ROOT / "real_data_checks_cuda.json")
from moe_transformer.data import validate_processed_data
metadata = validate_processed_data("data/processed/tinystories_10m_bpe4096")
print("Stories:", metadata["story_counts"])
print("Tokens:", {k: v["tokens"] for k, v in metadata["splits"].items()})

## Run the 200-update pilot

This uses the 5000-update schedule and FP16. Outputs go directly to Drive. Each run gets a new directory. If you already have a finished pilot, skip this cell and set PILOT to its path in the next cell.

In [ ]:
from datetime import datetime, timezone
def new_run(label):
    return DRIVE_ROOT / "runs" / (label + "_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"))

PILOT = new_run("D1_pilot")
run_module("moe_transformer.train", "--config", "configs/milestone.yaml",
           "--device", "cuda", "--run-dir", PILOT, "--stop-after", "200")
print("Pilot saved at:", PILOT)

In [ ]:
# To reuse an existing pilot after a restart, set PILOT = Path("...") here.
pilot_result = json.loads((PILOT / "results.json").read_text())
throughput = pilot_result["session_profile_tokens_per_second"]
print(json.dumps(pilot_result, indent=2))
print(f"Estimated remaining 5000-step TRAINING time: {(5000-200)*4096/throughput/60:.1f} minutes")
print("Allow additional time for development evaluations, checkpoint writes, and interruptions.")

## Train the main baseline

Leave HORIZON=5000 to resume the pilot. Set HORIZON=2500 if the shorter preplanned budget is needed; that starts a new run because the warmup/schedule differs. Karan must use the same selected horizon, data and training settings. This cell may take substantial time; follow the measured pilot estimate.

In [ ]:
HORIZON = 5000
assert HORIZON in (2500, 5000)
FINAL_RUN = new_run(f"D1_{HORIZON}")
if HORIZON == 5000:
    run_module("moe_transformer.train", "--config", "configs/milestone.yaml",
               "--device", "cuda", "--run-dir", FINAL_RUN, "--resume", PILOT / "last.pt")
else:
    run_module("moe_transformer.train", "--config", "configs/milestone_2500.yaml",
               "--device", "cuda", "--run-dir", FINAL_RUN)
print("Completed run:", FINAL_RUN)

## Optional: recover after a disconnected run

After rerunning setup, set RESUME_FROM to the run directory containing last.pt and best.pt. The saved config determines the original horizon. Set RESUME_FROM back to None for normal sequential execution.

In [ ]:
RESUME_FROM = None  # Example: DRIVE_ROOT / "runs" / "D1_5000_20261104_..."
if RESUME_FROM is not None:
    RESUME_FROM = Path(RESUME_FROM)
    FINAL_RUN = new_run("D1_resumed")
    run_module("moe_transformer.train", "--config", RESUME_FROM / "config.yaml",
               "--device", "cuda", "--run-dir", FINAL_RUN,
               "--resume", RESUME_FROM / "last.pt")

## Reproduce the selected checkpoint and export evidence

The evaluator uses the checkpoint's precision and the fixed development data. It must reproduce the saved best CE within expected numerical tolerance. Checkpoint count and status distinguish full training from a partial trial.

In [ ]:
run_module("moe_transformer.evaluate", "--checkpoint", FINAL_RUN / "best.pt",
           "--device", "cuda", "--output", FINAL_RUN / "reproduced_dev.json")
result = json.loads((FINAL_RUN / "results.json").read_text())
reproduced = json.loads((FINAL_RUN / "reproduced_dev.json").read_text())
assert abs(result["dev_ce"] - reproduced["ce"]) < 1e-4
assert result["status"] == "complete", "This is still a partial trial; resume before marking D1 complete."
PLOTS = DRIVE_ROOT / "figures" / FINAL_RUN.name
run_module("moe_transformer.plotting", FINAL_RUN, "--output", PLOTS)
from IPython.display import display, Image
display(Image(filename=str(PLOTS / "learning_curve.png")))
print(json.dumps(result, indent=2))
print("Results and checkpoints:", FINAL_RUN)

## Package the handoff

This creates a ZIP with the data/tokenizer, source, run checkpoints, logs, metrics, and figures. Give Karan this archive and the selected config. Update the milestone report with the actual measured results. Keep the ZIP and all parent runs in Drive.

In [ ]:
local_run = PROJECT / "runs" / FINAL_RUN.name
if not local_run.exists():
    shutil.copytree(FINAL_RUN, local_run)
local_figures = PROJECT / "artifacts/person1" / FINAL_RUN.name
shutil.copytree(PLOTS, local_figures, dirs_exist_ok=True)
shutil.copy2(DRIVE_ROOT / "real_data_checks_cuda.json", PROJECT / "artifacts/person1")
run_module("moe_transformer.handoff", "--run-dir", local_run.relative_to(PROJECT))
gpu_handoff = {
    "status": "CUDA dense baseline complete",
    "run": str(local_run.relative_to(PROJECT)),
    "results": result,
    "reproduced_development": reproduced,
    "acceptance": "Evaluate best.pt, then run one update from last.pt of a partial prefix using its saved config.",
}
(PROJECT / "artifacts/person1/gpu_handoff.json").write_text(json.dumps(gpu_handoff, indent=2))
(PROJECT / "handoff_gpu.md").write_text(
    "# Completed GPU handoff\n\n"
    + f"Run: {local_run.relative_to(PROJECT)}\n\n"
    + f"Updates: {result['completed_steps']}; exposures: {result['token_exposures']}\n\n"
    + f"Best step: {result['selected_checkpoint_step']}; dev CE: {result['dev_ce']}; "
    + f"dev perplexity: {result['dev_perplexity']}\n\n"
    + "Full machine-readable evidence: artifacts/person1/gpu_handoff.json\n"
)
shutil.copy2(PROJECT / "handoff_gpu.md", PROJECT / "artifacts/person1/handoff_gpu.md")
bundle = PROJECT / "output/person1_completed_handoff.zip"
bundle_args = [sys.executable, "scripts/build_bundle.py", "--output", str(bundle),
               "--include-run", str(local_run.relative_to(PROJECT))]
if "PILOT" in globals() and PILOT.exists():
    local_pilot = PROJECT / "runs" / PILOT.name
    if not local_pilot.exists():
        shutil.copytree(PILOT, local_pilot)
    bundle_args += ["--include-run", str(local_pilot.relative_to(PROJECT))]
subprocess.run(bundle_args, check=True)
shutil.copy2(bundle, DRIVE_ROOT / bundle.name)
files.download(str(bundle))